# Seat-level audit from a simple 3D model (Module 1)

This notebook is Module 1 of the Seat-Level Visibility Toolkit. It takes a simple geometric model of a room (walls, partitions, furniture, plants) and a list of seats, and for every seat it computes:

1. **Line of sight to plants** (pass / fail): the plant must be inside the seat's view angle, and at least one ray from the eye point to a sample point on the plant must be clear.
2. **GVI** (Green View Index): the share of a 60 × 60 grid of rays whose first hit is a plant.
3. **The share of seats that pass**, compared with the 75% threshold of WELL v2.1 M09 (plants/water route), and how the answer changes when the project's own settings change.

It runs in plain Python (numpy, pandas, matplotlib), with nothing to install. The first part uses a small toy room that matches the line-of-sight figure in the project document, so you can see how it works before you describe your own room.

**How to use it:** run the cells one at a time from top to bottom (▶ or Shift+Enter). The cells you will edit are the settings (Step 1) and the scene (Step 3).

**Before you rely on any result**
- Every default (eye height 1.2 m, 60° × 60° view, 3,600 rays, "any one clear ray") is a project assumption, not a value fixed by WELL or LEED.
- This is a research tool, not an official WELL or LEED checker. Which rooms and seats count, and how "direct line of sight" is defined, must follow the credit text you are using. The notebook only codes the WELL M09 plants/water seat test.
- Plants modeled as spheres or boxes give only an approximate GVI.
- LEED v5 Quality Views is based on floor area and views to the outdoors, and the WELL exterior-view route needs window geometry. **Neither is calculated here.**

## Step 1: Settings

| Setting | Meaning |
|---|---|
| `eye_height` | Seated eye height in meters (1.2 m is an assumption; set it from the credit text or state your choice) |
| `fov_h`, `fov_v` | Horizontal and vertical view angle in degrees (60° × 60°, an assumption; use the same values as in the photo notebook) |
| `n_h`, `n_v` | Number of columns and rows of the ray grid (60 × 60 = 3,600 rays) |
| `min_clear_rays` | How many clear rays to plant sample points a seat needs to pass (1 = any one clear ray) |
| `min_gvi_pct` | Optional extra condition: minimum GVI in % (0 = not used) |
| `threshold_pct` | Share of seats that must pass (75%, as in the WELL M09 text) |

The ray grid is laid out on a flat view plane in equal steps, like the pixels of a photo, so that this notebook and the photo notebook count the same share.

In [ ]:
params = dict(
    eye_height=1.2,
    fov_h=60,
    fov_v=60,
    n_h=60,
    n_v=60,
    min_clear_rays=1,
    min_gvi_pct=0.0,
    threshold_pct=75,
)
params

## Step 2: The calculation engine (no need to edit)

**Coordinates:** x and y are the floor plan in meters, z is height. A seat faces a direction given as an angle `facing_deg`: 0° = along +x, 90° = along +y (counter-clockwise when seen from above).

**Objects** are boxes (walls, partitions, desks, shelves) or spheres (plants), each with a *tag*:
- `solid`: blocks the view (opaque things).
- `vegetation`: an indoor plant. Counts in GVI and in the line-of-sight test. Plants never block the line of sight to other plants.
- `exterior`: outdoor greenery seen through a window. Counts in GVI only, not in the plant line-of-sight test (WELL M09 plants route is about plants in the space).
- `glass`: transparent. Rays pass through it (windows).

**How a seat is scored**
- *GVI:* rays are cast from the eye point through a grid on a flat view plane. For each ray, the nearest object that is not glass decides what the ray "sees". GVI = rays that hit vegetation ÷ all rays × 100.
- *Line of sight:* points are sampled on the surface of every plant. Points that face the seat and fall inside the view angle are tested with a straight line from the eye. A point is "clear" if no solid object crosses that line. The seat passes if the number of clear points is at least `min_clear_rays` (and GVI is at least `min_gvi_pct`).

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, Wedge

# ---------- objects ----------
def box(name, mn, mx, tag='solid'):
    return dict(name=name, type='box', tag=tag, mn=np.array(mn, float), mx=np.array(mx, float))

def sphere(name, c, r, tag='vegetation'):
    return dict(name=name, type='sphere', tag=tag, c=np.array(c, float), r=float(r))

def hit_t(o, D, obj):
    """Distance from origin o along each unit ray (rows of D) to the object; inf if the ray misses."""
    if obj['type'] == 'box':
        D = np.where(np.abs(D) < 1e-12, 1e-12, D)
        t1 = (obj['mn'] - o) / D
        t2 = (obj['mx'] - o) / D
        tmin = np.minimum(t1, t2).max(axis=1)
        tmax = np.maximum(t1, t2).min(axis=1)
        ok = (tmax >= np.maximum(tmin, 0)) & (tmax > 1e-9)
        t = np.where(tmin > 1e-9, tmin, tmax)
        return np.where(ok, t, np.inf)
    oc = o - obj['c']
    b = D @ oc
    c = oc @ oc - obj['r'] ** 2
    disc = b * b - c
    sq = np.sqrt(np.maximum(disc, 0))
    t = -b - sq
    t = np.where(t > 1e-9, t, -b + sq)
    ok = (disc >= 0) & (t > 1e-9)
    return np.where(ok, t, np.inf)

# ---------- view geometry ----------
def frame(yaw_deg):
    y = math.radians(yaw_deg)
    f = np.array([math.cos(y), math.sin(y), 0.0])      # forward
    r = np.array([math.sin(y), -math.cos(y), 0.0])     # right
    u = np.array([0.0, 0.0, 1.0])                      # up
    return f, r, u

def view_rays(yaw_deg, P):
    """Unit ray directions through an evenly spaced grid on a flat view plane."""
    f, r, u = frame(yaw_deg)
    a = ((np.arange(P['n_h']) + 0.5) / P['n_h'] * 2 - 1) * math.tan(math.radians(P['fov_h'] / 2))
    b = ((np.arange(P['n_v']) + 0.5) / P['n_v'] * 2 - 1) * math.tan(math.radians(P['fov_v'] / 2))
    A, B = np.meshgrid(a, b)
    D = f + A[..., None] * r + B[..., None] * u
    D = D.reshape(-1, 3)
    return D / np.linalg.norm(D, axis=1, keepdims=True), (P['n_v'], P['n_h'])

def gvi_at(o, yaw_deg, objects, P):
    """GVI (%) and the plant/not-plant grid for an eye point o facing yaw_deg."""
    D, shape = view_rays(yaw_deg, P)
    ts = np.stack([np.full(len(D), np.inf) if ob['tag'] == 'glass' else hit_t(o, D, ob)
                   for ob in objects], axis=1)
    idx = ts.argmin(axis=1)
    finite = np.isfinite(ts.min(axis=1))
    tags = np.array([ob['tag'] for ob in objects] + ['none'])
    veg = np.isin(tags[np.where(finite, idx, len(objects))], ['vegetation', 'exterior'])
    return float(veg.mean() * 100), veg.reshape(shape)

# ---------- line of sight ----------
def surface_samples(obj, n=40):
    """Sample points on an object's surface, with outward normals."""
    if obj['type'] == 'sphere':
        i = np.arange(n) + 0.5
        phi = np.arccos(1 - 2 * i / n)
        th = math.pi * (1 + 5 ** 0.5) * i
        N = np.stack([np.cos(th) * np.sin(phi), np.sin(th) * np.sin(phi), np.cos(phi)], axis=1)
        return obj['c'] + obj['r'] * N, N
    g = np.linspace(0.05, 0.95, 5)
    pts, nor = [], []
    for k in range(3):
        i, j = [m for m in range(3) if m != k]
        for side, sgn in ((obj['mn'][k], -1.0), (obj['mx'][k], 1.0)):
            for gi in g:
                for gj in g:
                    p = np.zeros(3)
                    p[k] = side
                    p[i] = obj['mn'][i] + gi * (obj['mx'][i] - obj['mn'][i])
                    p[j] = obj['mn'][j] + gj * (obj['mx'][j] - obj['mn'][j])
                    nv = np.zeros(3); nv[k] = sgn
                    pts.append(p); nor.append(nv)
    return np.array(pts), np.array(nor)

def los_check(o, yaw_deg, objects, P, tag='vegetation'):
    """Returns (clear sample points, plants with a clear point, plants inside the view angle)."""
    f, r, u = frame(yaw_deg)
    tan_h = math.tan(math.radians(P['fov_h'] / 2))
    tan_v = math.tan(math.radians(P['fov_v'] / 2))
    solids = [ob for ob in objects if ob['tag'] == 'solid']
    total, seen, inview = 0, [], []
    for ob in objects:
        if ob['tag'] != tag:
            continue
        pts, nor = surface_samples(ob)
        v = pts - o
        facing = np.einsum('ij,ij->i', nor, -v) > 0           # surface faces the seat
        fwd, right, up = v @ f, v @ r, v @ u
        inside = facing & (fwd > 1e-9) & (np.abs(right) <= tan_h * fwd) & (np.abs(up) <= tan_v * fwd)
        if not inside.any():
            continue
        inview.append(ob['name'])
        q = pts[inside]
        dist = np.linalg.norm(q - o, axis=1)
        D = (q - o) / dist[:, None]
        blocked = np.zeros(len(q), bool)
        for s in solids:
            blocked |= hit_t(o, D, s) < dist - 1e-6
        n_clear = int((~blocked).sum())
        if n_clear:
            seen.append(ob['name'])
            total += n_clear
    return total, seen, inview

# ---------- audit ----------
def audit(seats, objects, P):
    rows = []
    for s in seats:
        eh = s.get('eye_height', P['eye_height'])
        o = np.array([s['x'], s['y'], eh], float)
        gvi, _ = gvi_at(o, s['facing_deg'], objects, P)
        clear, seen, inview = los_check(o, s['facing_deg'], objects, P)
        ok = (clear >= P['min_clear_rays']) and (gvi >= P['min_gvi_pct'])
        rows.append(dict(seat_id=s['id'], zone=s.get('zone', 'all'), x=s['x'], y=s['y'],
                         facing_deg=s['facing_deg'], plants_in_view=len(inview),
                         plants_clear=len(seen), clear_points=clear,
                         gvi_pct=round(gvi, 1), los_pass=bool(ok)))
    return pd.DataFrame(rows)

def share_passing(df, P):
    n = len(df)
    need = math.ceil(P['threshold_pct'] / 100 * n)
    passed = int(df.los_pass.sum())
    return dict(seats=n, passed=passed, share_pct=round(passed / n * 100, 1) if n else float('nan'),
                needed=need, meets=bool(passed >= need) if n else False)

## Step 3: Describe the room (edit this for your own room)

The default scene is a small toy room that matches the line-of-sight figure in the project document:
- a room 8 m × 5 m × 2.8 m, with a window (glass) in the east wall and two trees outside;
- one plant (a sphere) and a partition 1.8 m high;
- three seats: **A** has a clear view to the plant, **B** faces the plant but the partition blocks it, **C** has the plant behind it.

**To describe your own room**, either edit the lists below, or set `USE_UPLOAD = True` and upload two CSV files:
- `seats.csv` with columns `id, x, y, facing_deg` (and optionally `zone`, `eye_height`)
- `objects.csv` with columns `name, type, tag, xmin, ymin, zmin, xmax, ymax, zmax` for boxes, or `name, type, tag, cx, cy, cz, r` for spheres (`type` is `box` or `sphere`; `tag` is `solid`, `vegetation`, `exterior` or `glass`)

Measure from a floor plan and a tape measure; the objects only need to be roughly right. Walls can be modeled as thin boxes, windows as `glass` boxes set into the wall, and the floor and ceiling as large flat boxes.

In [ ]:
USE_UPLOAD = False  #@param {type:"boolean"}

if USE_UPLOAD:
    from google.colab import files
    up = files.upload()
    seats_df = pd.read_csv('seats.csv')
    obj_df = pd.read_csv('objects.csv')
    seats = [{k: v for k, v in r_.items() if pd.notna(v)} for r_ in seats_df.to_dict('records')]
    objects = []
    for r_ in obj_df.to_dict('records'):
        if r_['type'] == 'box':
            objects.append(box(r_['name'], [r_['xmin'], r_['ymin'], r_['zmin']], [r_['xmax'], r_['ymax'], r_['zmax']], r_['tag']))
        else:
            objects.append(sphere(r_['name'], [r_['cx'], r_['cy'], r_['cz']], r_['r'], r_['tag']))
else:
    objects = [
        box('floor',      (-0.2, -0.2, -0.2), (8.2, 5.2, 0.0)),
        box('ceiling',    (-0.2, -0.2, 2.8),  (8.2, 5.2, 3.0)),
        box('west wall',  (-0.2, -0.2, 0.0),  (0.0, 5.2, 2.8)),
        box('south wall', (-0.2, -0.2, 0.0),  (8.2, 0.0, 2.8)),
        box('north wall', (-0.2, 5.0, 0.0),   (8.2, 5.2, 2.8)),
        # east wall with a window (y 1.0-4.0 m, z 0.9-2.4 m)
        box('east wall S',    (8.0, 0.0, 0.0), (8.2, 1.0, 2.8)),
        box('east wall N',    (8.0, 4.0, 0.0), (8.2, 5.0, 2.8)),
        box('east wall low',  (8.0, 1.0, 0.0), (8.2, 4.0, 0.9)),
        box('east wall high', (8.0, 1.0, 2.4), (8.2, 4.0, 2.8)),
        box('window glass',   (8.0, 1.0, 0.9), (8.2, 4.0, 2.4), 'glass'),
        # partition and plants
        box('partition', (3.0, 0.5, 0.0), (3.1, 2.6, 1.8)),
        sphere('plant 1', (4.5, 2.5, 0.9), 0.45),
        sphere('outdoor tree 1', (11.0, 2.0, 2.0), 1.5, 'exterior'),
        sphere('outdoor tree 2', (11.5, 3.8, 2.2), 1.6, 'exterior'),
    ]
    seats = [
        dict(id='A', x=1.0, y=3.5, facing_deg=0),
        dict(id='B', x=1.0, y=1.5, facing_deg=0),
        dict(id='C', x=5.8, y=2.5, facing_deg=0),
    ]
print(len(objects), 'objects,', len(seats), 'seats')

## Step 4: Score every seat

For each seat the table gives:
- `plants_in_view`: plants with at least one surface point inside the view angle
- `plants_clear`: plants with at least one clear sample point
- `clear_points`: number of clear sample points over all plants
- `gvi_pct`: Green View Index in %
- `los_pass`: the line-of-sight result under the settings above

Outdoor trees seen through a `glass` window (tagged `exterior`) add to `gvi_pct` but never make a seat pass the plant line-of-sight test. So seat C, which has the indoor plant behind it, fails even though its GVI is high.

In [ ]:
results = audit(seats, objects, params)
results

## Step 5: Plan view

A floor-plan picture of the result. Each seat shows its view angle as a wedge (green = pass, red = fail), its GVI, and its ID. Grey = solid objects, blue = glass, green circles = plants.

In [ ]:
def plot_plan(df, objects, P, ax=None, cone_len=2.5, title=None, xlim=None, ylim=None):
    ax = ax or plt.subplots(figsize=(9, 6))[1]
    for ob in objects:
        if ob['type'] == 'box':
            if ob['mx'][2] <= 0.01 or ob['mn'][2] >= 2.0:
                continue                           # skip floor and ceiling in plan
            col = {'glass': '#9ec9f0', 'vegetation': '#7fc97f', 'exterior': '#7fc97f'}.get(ob['tag'], '#b8b8b0')
            ax.add_patch(Rectangle(ob['mn'][:2], *(ob['mx'][:2] - ob['mn'][:2]), color=col, alpha=0.9, lw=0))
        else:
            col = '#7fc97f' if ob['tag'] in ('vegetation','exterior') else '#b8b8b0'
            ax.add_patch(Circle(ob['c'][:2], ob['r'], color=col, alpha=0.7, lw=0))
    for _, s in df.iterrows():
        col = '#2ca02c' if s.los_pass else '#d62728'
        ax.add_patch(Wedge((s.x, s.y), cone_len, s.facing_deg - P['fov_h'] / 2, s.facing_deg + P['fov_h'] / 2,
                           color=col, alpha=0.15, lw=0))
        ax.plot(s.x, s.y, 'o', color='black', ms=6)
        ax.text(s.x, s.y + 0.25, f"{s.seat_id}: {'pass' if s.los_pass else 'fail'}, GVI {s.gvi_pct}%",
                ha='center', fontsize=9)
    ax.set_aspect('equal')
    if xlim is None or ylim is None:               # fit the picture to the room that was loaded
        pts = [(ob['mn'][0], ob['mn'][1], ob['mx'][0], ob['mx'][1]) for ob in objects
               if ob['type'] == 'box' and ob['mx'][2] > 0.01 and ob['mn'][2] < 2.0 and ob['tag'] != 'exterior']
        pts += [(ob['c'][0] - ob['r'], ob['c'][1] - ob['r'], ob['c'][0] + ob['r'], ob['c'][1] + ob['r']) for ob in objects if ob['type'] != 'box']
        x0, y0 = min(p[0] for p in pts), min(p[1] for p in pts)
        x1, y1 = max(p[2] for p in pts), max(p[3] for p in pts)
        xlim = xlim or (x0 - 0.5, x1 + 0.5); ylim = ylim or (y0 - 0.5, y1 + 0.5)
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
    ax.set_title(title or 'Seat audit, plan view')
    return ax

plot_plan(results, objects, params)
plt.show()

## Step 6: GVI map across the room

This moves a seat across a grid of positions, all facing the same direction, and plots the GVI at each one. It shows where in the room a seat would have little green in view, which is also the starting point for the plant-placement notebook (Module 3).

Pick the direction from the dropdown in the cell below (no code to edit), then run the cell. Change the dropdown and run again to see another direction. `MAP_STEP` is the grid spacing in meters; smaller is slower. Positions inside solid objects are left blank.

In [ ]:
DIRECTION = "east"  #@param ["east", "north", "west", "south"]
MAP_STEP = 0.25  #@param {type:"number"}
MAP_FACING = {'east': 0, 'north': 90, 'west': 180, 'south': 270}[DIRECTION]

def inside_solid(x, y, z, objects):
    for ob in objects:
        if ob['tag'] == 'solid' and ob['type'] == 'box' and np.all(ob['mn'] <= [x, y, z]) and np.all([x, y, z] <= ob['mx']):
            return True
    return False

def gvi_map(objects, P, facing_deg, x_range, y_range, step):
    xs = np.arange(x_range[0], x_range[1] + 1e-9, step)
    ys = np.arange(y_range[0], y_range[1] + 1e-9, step)
    Z = np.full((len(ys), len(xs)), np.nan)
    for j, y in enumerate(ys):
        for i, x in enumerate(xs):
            if not inside_solid(x, y, P['eye_height'], objects):
                Z[j, i] = gvi_at(np.array([x, y, P['eye_height']]), facing_deg, objects, P)[0]
    return xs, ys, Z

# map the inside of the room: from the floor extent, 0.3 m in from the edges
_fl = [ob for ob in objects if ob['type'] == 'box' and ob['tag'] == 'solid' and ob['mx'][2] <= 0.01]
if _fl:
    _x0, _y0 = min(ob['mn'][0] for ob in _fl), min(ob['mn'][1] for ob in _fl)
    _x1, _y1 = max(ob['mx'][0] for ob in _fl), max(ob['mx'][1] for ob in _fl)
    MAP_X, MAP_Y = (max(_x0, 0) + 0.3, _x1 - 0.3), (max(_y0, 0) + 0.3, _y1 - 0.3)
else:
    MAP_X, MAP_Y = (0.3, 7.7), (0.3, 4.7)
xs, ys, Z = gvi_map(objects, params, MAP_FACING, MAP_X, MAP_Y, MAP_STEP)
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(Z, origin='lower', extent=(xs[0] - MAP_STEP / 2, xs[-1] + MAP_STEP / 2, ys[0] - MAP_STEP / 2, ys[-1] + MAP_STEP / 2), cmap='Greens')
plt.colorbar(im, label='GVI (%)')
for ob in objects:
    if ob['type'] == 'sphere' and ob['tag'] != 'exterior':
        ax.add_patch(Circle(ob['c'][:2], ob['r'], fill=False, ec='black', lw=1))
    if ob['type'] == 'box' and ob['tag'] == 'solid' and ob['mx'][2] > 0.01 and ob['mn'][2] < 2.0 and ob['mx'][0] < MAP_X[1] + 0.5 and ob['mn'][0] > MAP_X[0] - 0.5 and ob['mx'][1] < MAP_Y[1] + 0.5:
        ax.add_patch(Rectangle(ob['mn'][:2], *(ob['mx'][:2] - ob['mn'][:2]), fill=False, ec='black', lw=1))
ax.set_title(f'GVI of a seat at each position, facing {DIRECTION}')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
plt.show()

## Step 7: Share of seats that pass, and how it relates to WELL and LEED

The table below counts the seats that pass the line-of-sight test and compares the share with the threshold (WELL v2, Q2 2025, M09 Part 1 asks that at least 75% of workstations, conference room seats, classroom seats and seating within common spaces have a direct line of sight to indoor plants, water or nature views, or be within 33 ft of them).

**Counting rule used:** passing seats ÷ seats counted, with the number of seats needed rounded up (for example 10 seats need 8). The credit text does not say whether seats are pooled across rooms or counted room by room, and the text lists common spaces, so a library reading room is probably included, but this is a reading and should be stated. The notebook therefore shows both a pooled share and a share by `zone` (use the `zone` column in `seats.csv`).

**What is and is not covered:** the plant line-of-sight route, the WELL exterior-view route and LEED Quality Views are all calculated below, each with its own stated assumptions; see the summary table at the end of this step.

In [ ]:
pooled = share_passing(results, params)
print('Pooled over all seats:', pooled)

by_zone = pd.DataFrame([dict(zone=z, **share_passing(g, params)) for z, g in results.groupby('zone')])
print('\nBy zone:')
display(by_zone)

### WELL M09 exterior nature views (seats)

WELL M09 asks for *exterior nature views* within a direct line of sight of at least 75% of workstations and seats. Views must be natural areas or landscapes, such as green spaces (park, forest) or blue spaces (ocean, lake, river).

**How it is coded here:** the rays of the 60 × 60 view grid are traced from the seat. A ray counts as an exterior nature ray if the first thing it hits (glass is transparent) is an object tagged `exterior` (outdoor trees, a park or a lawn that you model outside the windows). A seat passes if at least `min_clear_rays` of its rays are exterior nature rays. Walls and furniture block rays, glass does not, so the window geometry is what decides which seats see out. (Rays are used here instead of sample points because the outdoor objects are very large.)

**What you must supply:** the notebook only knows about outdoor nature that is in `objects.csv` with tag `exterior`. If none is modeled, every seat fails this route. Whether what you modeled counts as a "natural area or landscape" is your judgement and has to be stated in the report.

In [ ]:
def exterior_rays(o, yaw_deg):
    D, _ = view_rays(yaw_deg, params)
    ts = np.stack([np.full(len(D), np.inf) if ob['tag'] == 'glass' else hit_t(o, D, ob) for ob in objects], axis=1)
    first = np.array([ob['tag'] for ob in objects] + ['none'])[np.where(np.isfinite(ts.min(axis=1)), ts.argmin(axis=1), len(objects))]
    return int((first == 'exterior').sum()), len(D)

ext_rows = []
for s in seats:
    o = np.array([s['x'], s['y'], s.get('eye_height', params['eye_height'])], float)
    n_ray, n_all = exterior_rays(o, s['facing_deg'])
    ext_rows.append(dict(seat_id=s['id'], zone=s.get('zone', 'all'), ext_rays=n_ray,
                         ext_view_pct=round(n_ray / n_all * 100, 1), los_pass=bool(n_ray >= params['min_clear_rays'])))
ext_results = pd.DataFrame(ext_rows)
ext_pooled = share_passing(ext_results, params)
n_ext = sum(1 for ob in objects if ob['tag'] == 'exterior')
print(f"Exterior nature objects in the model: {n_ext}")
if n_ext == 0:
    print("None are modeled (no object with tag 'exterior'), so this route cannot pass. Add outdoor trees, lawn or water to objects.csv to test it.")
print('Pooled over all seats:', ext_pooled)
display(pd.DataFrame([dict(zone=z, **share_passing(g, params)) for z, g in ext_results.groupby('zone')]))
display(ext_results)

### WELL v2 (Q2 2025) M09 Part 1 as written: three ways for a seat to pass

In the Q2 2025 text, Part 1 (Provide Nature Access Indoors) asks that at least 75% of workstations, conference room seats, classroom seats and seating within common spaces meet at least one of two requirements: (a) a direct line of sight to indoor plant(s), water feature(s) and/or nature view(s), or (b) being within 33 ft of indoor plant(s), water feature(s) and/or nature view(s).

**How it is coded here:** a seat passes Part 1 if any of three tests passes: line of sight to a plant (Step 4), line of sight to exterior nature (the exterior-view test above, which stands for "nature view"), or distance to the nearest plant of at most `WELL_DIST_FT` feet. Distance is the straight line from the seat's eye to the nearest point of the nearest plant, and walls are ignored because the text does not say how distance is measured. Water features are not modeled.

**Why both are shown:** distance in a small room is easily within 33 ft (10.06 m), so the distance test alone can pass every seat while some seats see almost no green. The line-of-sight and GVI results show what the yes/no result hides.

In [ ]:
WELL_DIST_FT = 33  #@param {type:"number"}
WELL_DIST_M = WELL_DIST_FT * 0.3048

def dist_to_object(o, ob):
    if ob['type'] == 'sphere':
        return max(float(np.linalg.norm(o - ob['c'])) - ob['r'], 0.0)
    d = np.maximum(np.maximum(ob['mn'] - o, 0), o - ob['mx'])
    return float(np.linalg.norm(d))

dist_rows = []
for s in seats:
    o = np.array([s['x'], s['y'], s.get('eye_height', params['eye_height'])], float)
    dm = min([dist_to_object(o, ob) for ob in objects if ob['tag'] == 'vegetation'] or [float('inf')])
    dist_rows.append(dict(seat_id=s['id'], zone=s.get('zone', 'all'), dist_to_plant_m=round(dm, 2),
                          los_pass=bool(dm <= WELL_DIST_M)))
dist_results = pd.DataFrame(dist_rows)

part1 = (results[['seat_id', 'zone', 'los_pass', 'gvi_pct']].rename(columns={'los_pass': 'plant_line_of_sight'})
         .merge(ext_results[['seat_id', 'los_pass']].rename(columns={'los_pass': 'exterior_view'}), on='seat_id')
         .merge(dist_results[['seat_id', 'dist_to_plant_m', 'los_pass']].rename(columns={'los_pass': 'within_distance'}), on='seat_id'))
part1['los_pass'] = part1.plant_line_of_sight | part1.exterior_view | part1.within_distance
dist_pooled = share_passing(dist_results, params)
part1_pooled = share_passing(part1, params)
print(f'Within {WELL_DIST_FT} ft of a plant:', dist_pooled)
print('Part 1, any of the three tests:', part1_pooled)
display(part1)

### LEED v5 EQc2 Option 1 Path 2: Quality Views (floor area)

LEED v5 asks for a view to the outdoors for 75% of regularly occupied floor area (2 points) or 90% (3 points). This is about **floor area**, not seats, so the notebook places a standing or seated eye point on a grid across the room and tests each point.

**Rules coded (from the LEED v5 BD+C text)**
- The view is a straight line from the eye point through a `glass` object (a window); it must not be blocked by any `solid` object on the way.
- The occupant must be within **3 × the head height of the glazing** of the window (the top of the glass box is taken as the head height).
- The view must contain **nature** (an `exterior` object) **or** nothing solid within **7.5 m** (25 ft) beyond the window. Urban landmarks and art are not coded.

**Not coded, so stated as assumptions:** glass visible light transmittance above 40% (all `glass` objects are assumed to qualify), the preserved view through frits and tints, views into atria, and which rooms are "regularly occupied". Use `LEED_EXCLUDE` to remove areas that are not regularly occupied (stores, plant rooms), as `x0,y0,x1,y1` boxes in meters separated by `;`. `LEED_EYE_HEIGHT` is the eye height of the occupant used for the area test.

In [ ]:
LEED_EYE_HEIGHT = 1.5  #@param {type:"number"}
LEED_EXCLUDE = ""  #@param {type:"string"}
LEED_STEP = 0.5  #@param {type:"number"}
HEAD_MULT, NEAR_LIMIT = 3.0, 7.5     # 3 x head height of the glazing; 7.5 m (25 ft) beyond the glass

def pane_points(g):
    """Sample points on the room-facing pane of a thin glass box."""
    k = int(np.argmin(g['mx'] - g['mn']))                    # thin axis = pane normal
    i, j = [m for m in range(3) if m != k]
    pts = []
    for gi in np.linspace(0.05, 0.95, 7):
        for gj in np.linspace(0.05, 0.95, 7):
            p = (g['mn'] + g['mx']) / 2
            p[i] = g['mn'][i] + gi * (g['mx'][i] - g['mn'][i])
            p[j] = g['mn'][j] + gj * (g['mx'][j] - g['mn'][j])
            pts.append(p)
    return np.array(pts), k

glass_boxes = [ob for ob in objects if ob['type'] == 'box' and ob['tag'] == 'glass']
solid_boxes = [ob for ob in objects if ob['tag'] == 'solid']
ext_objs = [ob for ob in objects if ob['tag'] == 'exterior']
panes = [(g, *pane_points(g)) for g in glass_boxes]

def has_quality_view(o):
    for g, pts, k in panes:
        dxy = np.linalg.norm((pts - o)[:, :2], axis=1)
        near = dxy <= HEAD_MULT * g['mx'][2]                 # within 3 x head height of the glazing
        if not near.any():
            continue
        q = pts[near]
        dist = np.linalg.norm(q - o, axis=1)
        D = (q - o) / dist[:, None]
        blocked = np.zeros(len(q), bool)
        for s_ in solid_boxes:
            blocked |= hit_t(o, D, s_) < dist - 1e-6          # something in the way before the glass
        ok = ~blocked
        if not ok.any():
            continue
        D2, d2 = D[ok], dist[ok]
        # beyond the glass: nature (an exterior object) counts; otherwise nothing solid within 7.5 m
        nature = np.zeros(len(D2), bool)
        for e in ext_objs:
            nature |= np.isfinite(hit_t(o, D2, e))
        near_obj = np.zeros(len(D2), bool)
        for s_ in solid_boxes:
            t = hit_t(o, D2, s_)
            near_obj |= (t > d2 + 1e-6) & (t < d2 + NEAR_LIMIT)
        if (nature | ~near_obj).any():
            return True
    return False

excl = parse_boxes = [[float(t) for t in part.replace(' ', '').split(',') if t != ''] for part in LEED_EXCLUDE.split(';') if part.strip()]
lx = np.arange(MAP_X[0], MAP_X[1] + 1e-9, LEED_STEP)
ly = np.arange(MAP_Y[0], MAP_Y[1] + 1e-9, LEED_STEP)
V = np.full((len(ly), len(lx)), np.nan)
for j, y in enumerate(ly):
    for i, x in enumerate(lx):
        if inside_solid(x, y, LEED_EYE_HEIGHT, objects) or any(a <= x <= c and b <= y <= d for a, b, c, d in excl):
            continue
        V[j, i] = 1.0 if has_quality_view(np.array([x, y, LEED_EYE_HEIGHT])) else 0.0

valid = ~np.isnan(V)
leed_share = float(np.nansum(V) / valid.sum() * 100) if valid.any() else float('nan')
print(f"Glass objects used as windows: {[g['name'] for g in glass_boxes]}")
print(f"Floor points tested: {int(valid.sum())} (every {LEED_STEP} m). With a quality view: {leed_share:.1f}% of the area")
print(f"  75% (2 points): {'meets' if leed_share >= 75 else 'does not meet'}    90% (3 points): {'meets' if leed_share >= 90 else 'does not meet'}")

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(V, origin='lower', extent=(lx[0] - LEED_STEP / 2, lx[-1] + LEED_STEP / 2, ly[0] - LEED_STEP / 2, ly[-1] + LEED_STEP / 2),
          cmap='RdYlGn', vmin=0, vmax=1)
for ob in solid_boxes:
    if ob['mx'][2] > 0.01 and ob['mn'][2] < 2.0:
        ax.add_patch(Rectangle(ob['mn'][:2], *(ob['mx'][:2] - ob['mn'][:2]), fill=False, ec='k', lw=0.6))
for g in glass_boxes:
    ax.add_patch(Rectangle(g['mn'][:2], *(g['mx'][:2] - g['mn'][:2]), color='#3b8ed0', lw=2))
ax.set_aspect('equal'); ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title(f'LEED Quality Views: green = view, red = no view ({leed_share:.0f}% of area)')
plt.show()

In [ ]:
credit_summary = pd.DataFrame([
    dict(credit='WELL M09 Part 1 as written: line of sight to plants, nature view, or within 33 ft (any of the three)',
         unit='seats / workstations', what_this_notebook_does='Coded: a seat passes if any of the three tests passes; water not modeled',
         result=f"{part1_pooled['passed']} of {part1_pooled['seats']} seats ({part1_pooled['share_pct']}%): {'meets' if part1_pooled['meets'] else 'does not meet'} 75%"),
    dict(credit='WELL M09 within 33 ft of a plant (distance only)',
         unit='seats / workstations', what_this_notebook_does='Coded: straight line from eye to nearest plant surface, walls ignored',
         result=f"{dist_pooled['passed']} of {dist_pooled['seats']} seats ({dist_pooled['share_pct']}%): {'meets' if dist_pooled['meets'] else 'does not meet'} 75%"),
    dict(credit='WELL M09: plants in direct line of sight (seats)',
         unit='seats / workstations', what_this_notebook_does='Coded: line of sight to plants per seat, share vs 75%',
         result=f"{pooled['passed']} of {pooled['seats']} seats ({pooled['share_pct']}%): {'meets' if pooled['meets'] else 'does not meet'} 75%"),
    dict(credit='WELL M09: exterior nature view (line of sight)',
         unit='seats / workstations', what_this_notebook_does="Coded: line of sight through glass to objects tagged 'exterior'",
         result=f"{ext_pooled['passed']} of {ext_pooled['seats']} seats ({ext_pooled['share_pct']}%): {'meets' if ext_pooled['meets'] else 'does not meet'} 75%"
                + ('' if n_ext else ' (no exterior nature modeled)')),
    dict(credit='LEED v5 EQc2 Quality Views',
         unit='regularly occupied floor area', what_this_notebook_does='Coded: view through glass within 3 x head height; nature or nothing within 7.5 m; glass assumed above 40% VLT',
         result=f"{leed_share:.1f}% of floor area: 75% {'met' if leed_share >= 75 else 'not met'}, 90% {'met' if leed_share >= 90 else 'not met'}"),
])
pd.set_option('display.max_colwidth', 140)
credit_summary

## Step 8: How much do the project's own settings matter?

This repeats the audit with different values of the settings that the standards do not fix, and shows how the share of passing seats changes. If a small change moves the result across the 75% line, the conclusion depends on the setting itself, and the report must say so.

In [ ]:
scenarios = [
    ('baseline', {}),
    ('view angle 35\u00b0 \u00d7 35\u00b0', dict(fov_h=35, fov_v=35)),
    ('view angle 90\u00b0 \u00d7 60\u00b0', dict(fov_h=90, fov_v=60)),
    ('eye height 1.1 m', dict(eye_height=1.1)),
    ('eye height 1.3 m', dict(eye_height=1.3)),
    ('need 5 clear points', dict(min_clear_rays=5)),
    ('need 20 clear points', dict(min_clear_rays=20)),
    ('also need GVI \u2265 1%', dict(min_gvi_pct=1.0)),
    ('also need GVI \u2265 5%', dict(min_gvi_pct=5.0)),
]
rows = []
for name, change in scenarios:
    P2 = {**params, **change}
    seats2 = [{k: v for k, v in s_.items() if not (k == 'eye_height' and 'eye_height' in change)} for s_ in seats]   # a changed eye height replaces each seat's own
    df2 = audit(seats2, objects, P2)
    sp = share_passing(df2, P2)
    rows.append(dict(scenario=name, seats_passing=sp['passed'], of_seats=sp['seats'],
                     share_pct=sp['share_pct'], meets_threshold=sp['meets'],
                     mean_gvi_pct=round(df2.gvi_pct.mean(), 1)))
pd.DataFrame(rows)

### Sensitivity of the WELL exterior-view route to the definition of "direct line of sight"

The WELL text does not say that a seat must face the view. The table compares three readings of the same model: the seat sees the outdoor nature inside its own 60° × 60° view (the default above), inside a wider 120° × 60° view, or from any direction (the seat is tested in twelve directions, 30° apart, and passes if any one of them sees it).

In [ ]:
def ext_pass(seat, fov_h, yaws):
    o = np.array([seat['x'], seat['y'], seat.get('eye_height', params['eye_height'])], float)
    P2 = {**params, 'fov_h': fov_h}
    best = 0
    for yw in yaws:
        D, _ = view_rays(yw, P2)
        ts = np.stack([np.full(len(D), np.inf) if ob['tag'] == 'glass' else hit_t(o, D, ob) for ob in objects], axis=1)
        first = np.array([ob['tag'] for ob in objects] + ['none'])[np.where(np.isfinite(ts.min(axis=1)), ts.argmin(axis=1), len(objects))]
        best = max(best, int((first == 'exterior').sum()))
    return best >= params['min_clear_rays']

readings = [
    ('facing, 60° × 60° (default)', 60, lambda s: [s['facing_deg']]),
    ('facing, 120° × 60°', 120, lambda s: [s['facing_deg']]),
    ('any direction (12 views of 60°)', 60, lambda s: list(range(0, 360, 30))),
]
rows = []
for name, fh, yaws in readings:
    df_ = pd.DataFrame([dict(seat_id=s['id'], zone=s.get('zone', 'all'), los_pass=ext_pass(s, fh, yaws(s))) for s in seats])
    sp = share_passing(df_, params)
    rows.append(dict(reading=name, seats_passing=sp['passed'], of_seats=sp['seats'], share_pct=sp['share_pct'], meets_threshold=sp['meets']))
pd.DataFrame(rows)

## Step 9: Validation checks (Module 1 against hand checks)

The project plan validates this tool in two ways.

**a) Hand check.** Check some seats by hand in the model (or on the floor plan): is the plant in view, is anything in the way? Enter your hand result for each seat in `MANUAL` (True = pass). The cell compares it with the tool, counts pass/fail disagreements, and computes Cohen's kappa. The values below are placeholders for the toy room: replace them with your own hand-checked seats.

**b) Ray-count convergence.** GVI should stop changing when the number of rays grows. The cell recomputes GVI with grids from 10 × 10 to 120 × 120 rays for the seats you list. If the value is still moving at 60 × 60, use more rays.

In [ ]:
MANUAL = {'A': True, 'B': False, 'C': False}   # toy-room placeholders: replace the keys with your own seat IDs (for example 'W3': True), True = pass

chk = results[results.seat_id.isin(MANUAL)][['seat_id', 'los_pass']].copy()
chk['hand'] = chk.seat_id.map(MANUAL)
chk['match'] = chk.los_pass == chk.hand
agree = chk.match.mean()
po = agree
p_tool, p_hand = chk.los_pass.mean(), chk.hand.mean()
pe = p_tool * p_hand + (1 - p_tool) * (1 - p_hand)
kappa = (po - pe) / (1 - pe) if pe < 1 else float('nan')
print(f'Seats checked: {len(chk)}   agreement: {agree * 100:.0f}%   '
      f'disagreements: {int((~chk.match).sum())}   Cohen\'s kappa: {kappa:.2f}')
display(chk)

CHECK_SEATS = [sid for sid in ['A', 'C'] if sid in set(results.seat_id)] or list(results.seat_id[:2])   # seats for the ray-count test
conv = []
for sid in CHECK_SEATS:
    s = next(x for x in seats if x['id'] == sid)
    o = np.array([s['x'], s['y'], s.get('eye_height', params['eye_height'])])
    for n in (10, 20, 40, 60, 90, 120):
        P2 = {**params, 'n_h': n, 'n_v': n}
        conv.append(dict(seat_id=sid, rays=n * n, gvi_pct=round(gvi_at(o, s['facing_deg'], objects, P2)[0], 2)))
pd.DataFrame(conv).pivot(index='rays', columns='seat_id', values='gvi_pct')

## Step 10: Save the results

This writes the seat table, the credit summary, the settings used (so that every parameter is on record), and the plan view, then downloads them as one zip file.

In [ ]:
import os, json, shutil
OUT_DIR = '/content/output_module1'
os.makedirs(OUT_DIR, exist_ok=True)
results.to_csv(os.path.join(OUT_DIR, 'seat_table.csv'), index=False)
credit_summary.to_csv(os.path.join(OUT_DIR, 'credit_summary.csv'), index=False)
json.dump(params, open(os.path.join(OUT_DIR, 'parameters_used.json'), 'w'), indent=2)
fig, ax = plt.subplots(figsize=(9, 6))
plot_plan(results, objects, params, ax=ax)
fig.savefig(os.path.join(OUT_DIR, 'plan_view.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
shutil.make_archive('/content/module1_results', 'zip', OUT_DIR)
try:
    from google.colab import files
    files.download('/content/module1_results.zip')
except Exception:
    print('Saved to /content/module1_results.zip')